In [2]:
import numpy as np
import pandas as pd
import sqlite3
from scipy import stats

df = pd.read_csv("IBM-HR-Employee-Attrition.csv")

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
#PHASE 1: Testing whether years with the current manager are statistically associated with performance outcomes.


In [6]:
phase1_counts = pd.read_sql_query("""
SELECT
    YearsWithCurrManager,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY YearsWithCurrManager, PerformanceRating
ORDER BY YearsWithCurrManager, PerformanceRating
""", conn)

manager_tenure_matrix = phase1_counts.pivot(
    index='YearsWithCurrManager',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

manager_tenure_matrix = manager_tenure_matrix.div(manager_tenure_matrix.sum(axis=1), axis=0) * 100

print("=== MANAGER TENURE ANALYSIS ===")
print(manager_tenure_matrix.round(1).astype(str) + "%")


=== MANAGER TENURE ANALYSIS ===
PerformanceRating          3      4
YearsWithCurrManager               
0                      82.9%  17.1%
1                      88.2%  11.8%
2                      88.1%  11.9%
3                      87.3%  12.7%
4                      78.6%  21.4%
5                      83.9%  16.1%
6                      89.7%  10.3%
7                      83.3%  16.7%
8                      78.5%  21.5%
9                      84.4%  15.6%
10                     81.5%  18.5%
11                     81.8%  18.2%
12                     88.9%  11.1%
13                     92.9%   7.1%
14                     80.0%  20.0%
15                     80.0%  20.0%
16                    100.0%   0.0%
17                     85.7%  14.3%


In [8]:
manager_mobility_contingency = pd.crosstab(df['YearsWithCurrManager'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(manager_mobility_contingency)

print("=== MOBILITY PHASE 1: MANAGER TENURE VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Years with current manager are statistically associated with performance outcomes.")
    print("Manager tenure appears related to differences in performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years with current manager.")
    print("This dataset does not provide strong evidence that manager tenure is associated with performance outcomes.")


=== MOBILITY PHASE 1: MANAGER TENURE VALIDATION ===
Chi-Square Statistic : 13.8433
Degrees of Freedom   : 17
P-Value              : 0.678165

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years with current manager.
This dataset does not provide strong evidence that manager tenure is associated with performance outcomes.


In [10]:
#PHASE 2: Testing whether years in current role are statistically associated with performance outcomes.


In [12]:
phase2_counts = pd.read_sql_query("""
SELECT
    YearsInCurrentRole,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY YearsInCurrentRole, PerformanceRating
ORDER BY YearsInCurrentRole, PerformanceRating
""", conn)

role_tenure_matrix = phase2_counts.pivot(
    index='YearsInCurrentRole',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

role_tenure_matrix = role_tenure_matrix.div(role_tenure_matrix.sum(axis=1), axis=0) * 100

print("=== ROLE TENURE ANALYSIS ===")
print(role_tenure_matrix.round(1).astype(str) + "%")


=== ROLE TENURE ANALYSIS ===
PerformanceRating        3      4
YearsInCurrentRole               
0                    84.0%  16.0%
1                    87.7%  12.3%
2                    87.6%  12.4%
3                    84.4%  15.6%
4                    81.7%  18.3%
5                    72.2%  27.8%
6                    86.5%  13.5%
7                    85.6%  14.4%
8                    83.1%  16.9%
9                    83.6%  16.4%
10                   86.2%  13.8%
11                   81.8%  18.2%
12                   70.0%  30.0%
13                   71.4%  28.6%
14                   81.8%  18.2%
15                   75.0%  25.0%
16                   71.4%  28.6%
17                  100.0%   0.0%
18                  100.0%   0.0%


In [14]:
role_mobility_contingency = pd.crosstab(df['YearsInCurrentRole'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(role_mobility_contingency)

print("=== MOBILITY PHASE 2: ROLE TENURE VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Years in current role are statistically associated with performance outcomes.")
    print("Role tenure appears related to differences in performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years in current role.")
    print("This dataset does not provide strong evidence that role tenure is associated with performance outcomes.")


=== MOBILITY PHASE 2: ROLE TENURE VALIDATION ===
Chi-Square Statistic : 14.8397
Degrees of Freedom   : 18
P-Value              : 0.672946

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years in current role.
This dataset does not provide strong evidence that role tenure is associated with performance outcomes.


In [16]:
#PHASE 3: Testing whether years since last promotion is statistically associated with attrition risk.


In [18]:
phase3_counts = pd.read_sql_query("""
SELECT
    YearsSinceLastPromotion,
    Attrition,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY YearsSinceLastPromotion, Attrition
ORDER BY YearsSinceLastPromotion, Attrition
""", conn)

promotion_velocity_matrix = phase3_counts.pivot(
    index='YearsSinceLastPromotion',
    columns='Attrition',
    values='EmployeeCount'
).fillna(0)

promotion_velocity_matrix = promotion_velocity_matrix.div(promotion_velocity_matrix.sum(axis=1), axis=0) * 100

print("=== CAREER PROGRESSION ANALYSIS ===")
print(promotion_velocity_matrix.round(1).astype(str) + "%")


=== CAREER PROGRESSION ANALYSIS ===
Attrition                    No    Yes
YearsSinceLastPromotion               
0                         81.1%  18.9%
1                         86.3%  13.7%
2                         83.0%  17.0%
3                         82.7%  17.3%
4                         91.8%   8.2%
5                         95.6%   4.4%
6                         81.2%  18.8%
7                         78.9%  21.1%
8                        100.0%   0.0%
9                         76.5%  23.5%
10                        83.3%  16.7%
11                        91.7%   8.3%
12                       100.0%   0.0%
13                        80.0%  20.0%
14                        88.9%  11.1%
15                        76.9%  23.1%


In [20]:
promotion_contingency = pd.crosstab(df['YearsSinceLastPromotion'], df['Attrition'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(promotion_contingency)

print("=== MOBILITY PHASE 3: CAREER PROGRESSION VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Years since last promotion are statistically associated with attrition risk.")
    print("Promotion stagnation appears related to differences in attrition outcomes.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: Attrition rates do not differ significantly by years since last promotion.")
    print("This dataset does not provide strong evidence that promotion timing is associated with attrition risk.")


=== MOBILITY PHASE 3: CAREER PROGRESSION VALIDATION ===
Chi-Square Statistic : 21.8450
Degrees of Freedom   : 15
P-Value              : 0.111934

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: Attrition rates do not differ significantly by years since last promotion.
This dataset does not provide strong evidence that promotion timing is associated with attrition risk.
